# OpenPlaque — LCX Local AV-Junction Ribbon (Robust)
Same experiment branch; runtime/setup repair with persistent traceback logging.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/LCX_Local_AV_Junction_Ribbon_v1_robust'
BRANCH = 'lcx-local-av-junction-from-main'
print('Branch:', BRANCH)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil, subprocess, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo,'pytest','SimpleITK','scipy','pandas','matplotlib'],check=True)
print('HEAD:',subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-c',"import openplaque; from openplaque.lcx_local_av_junction_identity_robust import ALGORITHM; from openplaque.lcx_local_av_junction_identity import synthetic_local_junction_self_test; print(openplaque.__file__); print(ALGORITHM); print(synthetic_local_junction_self_test())"],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','/content/OpenPlaque/tests/test_lcx_local_av_junction_identity.py'],check=True)


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_left.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_left.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_right.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_right.nii.gz',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_myocardium.nii.gz',root/'Joint_Three_Vessel_Template_Classifier_v1/LCX_joint_candidate_ranking.csv']+[root/f'Joint_Three_Vessel_Template_Classifier_v1/candidate_{i:02d}_source_path.csv' for i in range(1,6)]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import os, subprocess, sys
runner=f'''\nfrom openplaque.lcx_local_av_junction_identity_robust import run\nr=run(r"{DRIVE_ROOT}", r"{OUTPUT_DIR}")\nprint("STATUS:",r["summary"]["status"])\nprint("CONTROLS:",r["summary"]["local_junction_controls"])\nprint("DECISION:",r["summary"]["decision"])\nprint("REPORT:",r["report"])\nprint("ZIP:",r["zip"])\n'''
env=os.environ.copy(); env.update({'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','NUMEXPR_NUM_THREADS':'1'})
cp=subprocess.run([sys.executable,'-u','-c',runner],env=env,text=True,capture_output=True)
print(cp.stdout)
if cp.stderr: print('CHILD STDERR:\n'+cp.stderr)
if cp.returncode != 0:
    print('Persistent traceback, if available:', OUTPUT_DIR + '/failure_traceback.txt')
    raise RuntimeError(f'Analysis subprocess failed with return code {cp.returncode}')
